# Exercise 5 Solution: Real-World Analysis Project

Complete solutions for the AdventureWorks business analysis.

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

conn = duckdb.connect()

## Part 1: Load and Prepare Data

In [ ]:
# Solution: Load all tables
conn.execute("""
    CREATE TABLE products AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.Product.csv');
    CREATE TABLE product_categories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductCategory.csv');
    CREATE TABLE product_subcategories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Production.ProductSubcategory.csv');
    CREATE TABLE sales_orders AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderHeader.csv');
    CREATE TABLE sales_details AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesOrderDetail.csv');
    CREATE TABLE territories AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.SalesTerritory.csv');
    CREATE TABLE customers AS SELECT * FROM read_csv_auto('../sample_data/AW_CSV/Sales.Customer.csv');
""")
print("✓ All tables loaded")
conn.execute("SHOW TABLES").df()

## Part 2: Check Data Quality

In [ ]:
# Solution: Data Quality
print("=== NULL Check ===")
print(conn.execute("SELECT COUNT(*) - COUNT(CustomerID) AS null_customer FROM sales_orders").df())
print("\n=== Duplicate Check ===")
print(conn.execute("SELECT COUNT(*) AS total, COUNT(DISTINCT SalesOrderID) AS unique_orders FROM sales_orders").df())

## Part 3: KPI 1 - Revenue Overview
### 3.1 Total Revenue

In [ ]:
conn.execute("""
    SELECT COUNT(DISTINCT SalesOrderID) AS orders, ROUND(SUM(TotalDue), 2) AS revenue, ROUND(AVG(TotalDue), 2) AS avg_order
    FROM sales_orders
""").df()

### 3.2 Monthly Revenue

In [ ]:
conn.execute("""
    SELECT DATE_TRUNC('month', OrderDate)::DATE AS month, COUNT(*) AS orders, ROUND(SUM(TotalDue), 2) AS revenue
    FROM sales_orders GROUP BY 1 ORDER BY 1
""").df()

### 3.3 Year-over-Year Growth

In [ ]:
conn.execute("""
    WITH yearly AS (SELECT YEAR(OrderDate) AS year, SUM(TotalDue) AS revenue FROM sales_orders GROUP BY 1)
    SELECT year, ROUND(revenue, 2) AS revenue,
           ROUND(100.0 * (revenue - LAG(revenue) OVER (ORDER BY year)) / LAG(revenue) OVER (ORDER BY year), 1) AS yoy_pct
    FROM yearly ORDER BY year
""").df()

### 3.4 Revenue Trend Visualization

In [ ]:
df = conn.execute("SELECT DATE_TRUNC('month', OrderDate)::DATE AS month, SUM(TotalDue) AS revenue FROM sales_orders GROUP BY 1 ORDER BY 1").df()
fig = px.line(df, x='month', y='revenue', title='Monthly Revenue Trend')
fig.show()

## Part 4: KPI 2 - Product Performance
### 4.1 Top 10 Products

In [ ]:
conn.execute("""
    SELECT p.Name, SUM(sd.OrderQty) AS units, ROUND(SUM(sd.LineTotal), 2) AS revenue
    FROM sales_details sd JOIN products p ON sd.ProductID = p.ProductID
    GROUP BY p.Name ORDER BY revenue DESC LIMIT 10
""").df()

### 4.2 Category Analysis

In [ ]:
conn.execute("""
    SELECT pc.Name AS category, COUNT(DISTINCT sd.SalesOrderID) AS orders, SUM(sd.OrderQty) AS units, ROUND(SUM(sd.LineTotal), 2) AS revenue
    FROM sales_details sd
    JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name ORDER BY revenue DESC
""").df()

### 4.3 Category Visualization

In [ ]:
df = conn.execute("""
    SELECT pc.Name AS category, SUM(sd.LineTotal) AS revenue
    FROM sales_details sd JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID
    GROUP BY pc.Name
""").df()
fig = px.pie(df, values='revenue', names='category', title='Revenue by Category')
fig.show()

## Part 5: KPI 3 - Geography
### 5.1 Revenue by Territory

In [ ]:
conn.execute("""
    SELECT t.Name AS territory, t.\"Group\" AS region, COUNT(*) AS orders, ROUND(SUM(so.TotalDue), 2) AS revenue
    FROM sales_orders so JOIN territories t ON so.TerritoryID = t.TerritoryID
    GROUP BY t.Name, t.\"Group\" ORDER BY revenue DESC
""").df()

### 5.2 Regional Performance

In [ ]:
conn.execute("""
    SELECT t.\"Group\" AS region, YEAR(so.OrderDate) AS year, ROUND(SUM(so.TotalDue), 2) AS revenue
    FROM sales_orders so JOIN territories t ON so.TerritoryID = t.TerritoryID
    GROUP BY 1, 2 ORDER BY 1, 2
""").df()

### 5.3 Territory Visualization

In [ ]:
df = conn.execute("""
    SELECT t.Name AS territory, SUM(so.TotalDue) AS revenue
    FROM sales_orders so JOIN territories t ON so.TerritoryID = t.TerritoryID
    GROUP BY t.Name ORDER BY revenue DESC
""").df()
fig = px.bar(df, x='territory', y='revenue', title='Revenue by Territory')
fig.update_xaxes(tickangle=45)
fig.show()

## Part 6: KPI 4 - Customer Insights
### 6.1 Average Basket Value

In [ ]:
conn.execute("""
    SELECT ROUND(AVG(TotalDue), 2) AS avg_basket, ROUND(MEDIAN(TotalDue), 2) AS median_basket,
           ROUND(MIN(TotalDue), 2) AS min_basket, ROUND(MAX(TotalDue), 2) AS max_basket
    FROM sales_orders
""").df()

### 6.2 Customer Distribution

In [ ]:
conn.execute("""
    WITH customer_orders AS (SELECT CustomerID, COUNT(*) AS order_count FROM sales_orders GROUP BY 1)
    SELECT CASE WHEN order_count = 1 THEN '1 order' WHEN order_count <= 5 THEN '2-5 orders'
                WHEN order_count <= 10 THEN '6-10 orders' ELSE '10+ orders' END AS segment,
           COUNT(*) AS customer_count
    FROM customer_orders GROUP BY 1 ORDER BY customer_count DESC
""").df()

### 6.3 Pareto Analysis

In [ ]:
conn.execute("""
    WITH customer_revenue AS (
        SELECT CustomerID, SUM(TotalDue) AS revenue FROM sales_orders GROUP BY 1
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (ORDER BY revenue DESC) AS rn, COUNT(*) OVER () AS total_customers,
               SUM(revenue) OVER () AS total_revenue
        FROM customer_revenue
    )
    SELECT 'Top 20%' AS segment, COUNT(*) AS customers, ROUND(SUM(revenue), 2) AS revenue,
           ROUND(100.0 * SUM(revenue) / MAX(total_revenue), 1) AS pct
    FROM ranked WHERE rn <= total_customers * 0.2
    UNION ALL
    SELECT 'Bottom 80%', COUNT(*), ROUND(SUM(revenue), 2), ROUND(100.0 * SUM(revenue) / MAX(total_revenue), 1)
    FROM ranked WHERE rn > total_customers * 0.2
""").df()

## Part 7: Advanced Analysis
### 7.1 Product Affinity

In [ ]:
conn.execute("""
    WITH order_products AS (
        SELECT sd.SalesOrderID, p.Name FROM sales_details sd JOIN products p ON sd.ProductID = p.ProductID
    )
    SELECT a.Name AS product_1, b.Name AS product_2, COUNT(*) AS bought_together
    FROM order_products a JOIN order_products b ON a.SalesOrderID = b.SalesOrderID AND a.Name < b.Name
    GROUP BY 1, 2 HAVING COUNT(*) > 50 ORDER BY 3 DESC LIMIT 15
""").df()

### 7.2 Seasonality

In [ ]:
print("Monthly:")
print(conn.execute("SELECT MONTH(OrderDate) AS month, ROUND(AVG(TotalDue), 2) AS avg_revenue, COUNT(*) AS orders FROM sales_orders GROUP BY 1 ORDER BY 1").df())
print("\nDay of Week:")
print(conn.execute("SELECT DAYOFWEEK(OrderDate) AS dow, ROUND(AVG(TotalDue), 2) AS avg_revenue, COUNT(*) AS orders FROM sales_orders GROUP BY 1 ORDER BY 1").df())

### 7.3 Price Elasticity

In [ ]:
conn.execute("""
    SELECT CASE WHEN UnitPrice < 50 THEN 'Budget' WHEN UnitPrice < 200 THEN 'Mid-range'
                WHEN UnitPrice < 1000 THEN 'Premium' ELSE 'Luxury' END AS price_segment,
           COUNT(*) AS transactions, SUM(OrderQty) AS units, ROUND(SUM(LineTotal), 2) AS revenue
    FROM sales_details GROUP BY 1 ORDER BY revenue DESC
""").df()

## Part 8: Executive Dashboard

In [ ]:
# Get data
monthly_df = conn.execute("SELECT DATE_TRUNC('month', OrderDate)::DATE AS month, SUM(TotalDue) AS revenue FROM sales_orders GROUP BY 1 ORDER BY 1").df()
category_df = conn.execute("""
    SELECT pc.Name AS category, SUM(sd.LineTotal) AS revenue
    FROM sales_details sd JOIN products p ON sd.ProductID = p.ProductID
    JOIN product_subcategories ps ON p.ProductSubcategoryID = ps.ProductSubcategoryID
    JOIN product_categories pc ON ps.ProductCategoryID = pc.ProductCategoryID GROUP BY 1
""").df()
territory_df = conn.execute("SELECT t.Name AS territory, SUM(so.TotalDue) AS revenue FROM sales_orders so JOIN territories t ON so.TerritoryID = t.TerritoryID GROUP BY 1 ORDER BY 2 DESC LIMIT 5").df()

# Dashboard
fig = make_subplots(rows=2, cols=2, subplot_titles=('Monthly Revenue', 'Categories', 'Top Territories', 'Metrics'),
                    specs=[[{'type': 'scatter'}, {'type': 'pie'}], [{'type': 'bar'}, {'type': 'table'}]])
fig.add_trace(go.Scatter(x=monthly_df['month'], y=monthly_df['revenue'], mode='lines+markers'), row=1, col=1)
fig.add_trace(go.Pie(labels=category_df['category'], values=category_df['revenue']), row=1, col=2)
fig.add_trace(go.Bar(x=territory_df['territory'], y=territory_df['revenue']), row=2, col=1)
metrics = conn.execute("""
    SELECT 'Total Revenue' AS metric, '$' || ROUND(SUM(TotalDue)/1000000, 1) || 'M' AS value FROM sales_orders
    UNION ALL SELECT 'Total Orders', CAST(COUNT(*) AS VARCHAR) FROM sales_orders
    UNION ALL SELECT 'Avg Order Value', '$' || CAST(ROUND(AVG(TotalDue), 0) AS VARCHAR) FROM sales_orders
""").df()
fig.add_trace(go.Table(header=dict(values=['Metric', 'Value']), cells=dict(values=[metrics['metric'], metrics['value']])), row=2, col=2)
fig.update_layout(height=700, title_text='Executive Dashboard', showlegend=False)
fig.show()

## Part 9: Export Data

In [ ]:
# Export
conn.execute("COPY (SELECT DATE_TRUNC('month', OrderDate)::DATE AS month, SUM(TotalDue) AS revenue FROM sales_orders GROUP BY 1 ORDER BY 1) TO '../exports/monthly_summary.csv' (HEADER)")
print("✓ Exported monthly_summary.csv")
conn.execute("COPY (SELECT p.Name, SUM(sd.LineTotal) AS revenue FROM sales_details sd JOIN products p ON sd.ProductID = p.ProductID GROUP BY 1 ORDER BY 2 DESC) TO '../exports/product_performance.parquet' (FORMAT PARQUET)")
print("✓ Exported product_performance.parquet")

## Part 10: Management Summary

#### Top 3 Insights:
1. **Bikes dominate** - Highest revenue category
2. **Strong YoY growth** - Consistent growth with Q2/Q4 peaks
3. **Pareto effect** - Top 20% customers = ~80% revenue

#### Top 3 Recommendations:
1. **VIP program** for top customers
2. **Expand** underperforming territories
3. **Bundle products** based on affinity data

#### Next Steps:
- Build CLV prediction model
- Automate monthly reporting
- Develop churn prediction

## 🎉 Project Completed!

You've completed a full analysis project with DuckDB!